[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhmk/hocmaytrongtaichinh/blob/main/Module_6/L2.ipynb)

## HỌC MÁY TRONG TÀI CHÍNH

MODULE 6 | BÀI 2


---

# **ỨNG DỤNG HỌC SÂU TRONG DỰ BÁO ĐƯỜNG CONG LỢI SUẤT**

|  |  |
|:---|:---|
|**Thời gian đọc** |  100 phút |
|**Kiến thức nền** | Học máy, giới thiệu về học sâu  |
|**Từ khóa** |Đường cong lợi suất, hồi quy logistic, perceptron nhiều lớp |


---

In [ ]:
# Chuẩn bị môi trường trên Colab (chạy ô này một lần trước khi chạy các ô khác)
%pip install -q nnv pandas-datareader
import os, urllib.request, urllib.parse

BASE = "https://raw.githubusercontent.com/thanhmk/hocmaytrongtaichinh/main/Module_6/"

*Ở bài trước, ta đã giới thiệu khái niệm học sâu và các ứng dụng của nó. Bài này sẽ xây dựng tiếp trên các khái niệm học sâu bằng cách giải một bài toán thực tế là dự báo đường cong lợi suất bằng các thuật toán học sâu và so sánh kết quả với hồi quy đa thức bậc cao.*

## **1. Nhắc lại về đường cong lợi suất**
Ta có thể định nghĩa đường cong lợi suất là đồ thị thể hiện lãi suất (của trái phiếu) ở các kỳ hạn khác nhau. Đường cong lợi suất quan trọng trong thế giới tài chính vì các lý do sau:

1. Chúng được dùng làm chuẩn để đặt lãi suất thế chấp, và;
2. Chúng là chỉ báo tốt để dự báo tăng trưởng kinh tế.

Có ba loại đường cong lợi suất:

- Đường cong lợi suất bình thường
- Đường cong lợi suất đảo ngược
- Đường cong lợi suất phẳng

Ta sẽ xem lại từng loại bên dưới.

### **1.1 Đường cong lợi suất bình thường**

Đồ thị của đường cong lợi suất bình thường có xu hướng đi lên, hàm ý trái phiếu có ngày đáo hạn dài hơn sẽ trả lãi nhiều hơn (có lợi suất cao hơn) so với trái phiếu có ngày đáo hạn ngắn hơn.

Đường cong lợi suất bình thường có thể diễn giải là mọi người tin rằng nền kinh tế sẽ hoạt động tốt trong tương lai, và do đó ta kỳ vọng các ngân hàng trung ương sẽ tăng lãi suất vào thời điểm tương lai đó. Các ngân hàng trung ương tăng lãi suất để đáp lại áp lực lạm phát do nền kinh tế mở rộng gây ra.

### **1.2 Đường cong lợi suất đảo ngược**

Đường cong lợi suất đảo ngược là đường cong dốc xuống, hàm ý một cuộc suy thoái sắp xảy ra. Đường cong lợi suất đảo ngược nghĩa là mọi người tin rằng lãi suất sẽ thấp hơn hiện tại, và do đó ta kỳ vọng các ngân hàng trung ương sẽ cắt giảm lãi suất để khuyến khích nhà đầu tư, người tiêu dùng và doanh nghiệp vay thêm tiền. Điều này lại dẫn đến tăng trưởng kinh tế.

### **1.3 Đường cong lợi suất phẳng**

Ta có đường cong lợi suất phẳng khi nền kinh tế đang chuyển từ trạng thái này sang trạng thái khác. Ví dụ, khi một nền kinh tế đang mở rộng bắt đầu thu hẹp, ta thấy đường cong lợi suất ban đầu bình thường bắt đầu phẳng đi do mọi người kỳ vọng kinh tế suy giảm và do đó lãi suất dài hạn giảm. Điều ngược lại cũng đúng.

Bây giờ ta đã nắm đường cong lợi suất hữu ích thế nào, ở các phần tiếp theo ta sẽ dự báo đường cong lợi suất bằng các thuật toán học máy thông thường và một thuật toán học sâu.

## **2. Dự báo đường cong lợi suất**
Nghiên cứu tình huống này là bài toán chuỗi thời gian, mà nền tảng của nó thuộc về kinh tế lượng. Đây cũng là bài toán học máy có giám sát vì ta có các đầu ra để mô hình học. Biến dự báo trong trường hợp này sẽ là nhiều đầu ra.

Để giải bài toán này, ta sẽ theo một phương pháp luận chuẩn của ngành: Quy trình liên ngành cho khai phá dữ liệu (CRISP-DM), dùng để giải các bài toán khoa học dữ liệu.

Trừ giai đoạn triển khai, dành cho các bạn tò mò tự thực hành, các bước còn lại sẽ được minh họa ở đây.

### **2.1 Hiểu bài toán**
Trọng tâm của nghiên cứu này là tìm và hiểu các yếu tố ảnh hưởng đến biến động của đường cong lợi suất. Trong các biến ta sẽ dùng có dữ liệu lịch sử của đường cong lợi suất, tỷ lệ phần trăm nợ liên bang do nhà đầu tư nắm giữ, và chênh lệch lãi suất doanh nghiệp. Biến phụ thuộc sẽ là kỳ hạn 1 tháng, 5 năm và 30 năm của đường cong lợi suất.

Ta lấy dữ liệu từ các trang Yahoo và FRED. Trong bài toán này, ta sẽ dùng các đa thức có bậc khác nhau để dự báo đường cong lợi suất.

### **2.2 Hiểu dữ liệu**
Ta bắt đầu bằng việc nạp các gói sẽ dùng trong suốt bài này như bên dưới.

In [ ]:
# Load libraries
# Global Libraries
# Disable the warnings
import warnings

# Time series Models
from datetime import datetime

import numpy as np
import pandas as pd
import pandas_datareader.data as web

# Plotting
import seaborn as sns

# Libraries for Statistical Models
import statsmodels.api as sm
from matplotlib import pyplot
from pandas.plotting import scatter_matrix

# Regression ML
from sklearn.linear_model import LinearRegression

# Loss Function
from sklearn.metrics import mean_squared_error

# Modeling
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    cross_val_score,
    train_test_split,
)
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import PolynomialFeatures

warnings.filterwarnings("ignore")

Tiếp theo, ta nạp dữ liệu bằng hàm `datareader` của pandas:

In [ ]:
# downloading the data
start = datetime(2007, 6, 30)
end = datetime(2022, 6, 30)

data = [
    "DGS1MO",
    "DGS3MO",
    "DGS1",
    "DGS2",
    "DGS5",
    "DGS7",
    "DGS10",
    "DGS30",
    "TREAST",  # -- U.S. Treasury securities held by the Federal Reserve ( Millions of Dollars )
    "FYGFDPUN",  # -- Federal Debt Held by the Public ( Millions of Dollars )
    "FDHBFIN",  # -- Federal Debt Held by Foreign and International Investors ( Billions of Dollars )
    "GFDEBTN",  # -- Federal Debt: Total Public Debt ( Millions of Dollars )
    "BAA10Y",  # -- Baa Corporate Bond Yield Relative to Yield on 10-Year
]
data = web.DataReader(data, "fred", start, end).dropna(how="all").ffill()
data["FDHBFIN"] = data["FDHBFIN"] * 1000
data["GOV_PCT"] = data["TREAST"] / data["GFDEBTN"]
data["HOM_PCT"] = data["FYGFDPUN"] / data["GFDEBTN"]
data["FOR_PCT"] = data["FDHBFIN"] / data["GFDEBTN"]

Sau đó ta định nghĩa các biến độc lập và phụ thuộc như đã nêu ở mục trước, và ta cũng giả định rằng giao dịch chỉ diễn ra vào các ngày trong tuần. Do đó, ta sẽ tính phiên bản trễ của các biến dùng độ trễ 5 ngày giao dịch.

In [ ]:
data.rename(
    columns={
        "DGS1MO": "1m",
        "DGS3MO": "3m",
        "DGS1": "1y",
        "DGS2": "2y",
        "DGS5": "5y",
        "DGS7": "7y",
        "DGS10": "10y",
        "DGS30": "30y",
    },
    inplace=True,
)
return_period = 5

Y = data.loc[:, ["1m", "5y", "30y"]].shift(-return_period)
Y.columns = [col + "_pred" for col in Y.columns]

X = data.loc[
    :,
    [
        "1m",
        "3m",
        "1y",
        "2y",
        "5y",
        "7y",
        "10y",
        "30y",
        "GOV_PCT",
        "HOM_PCT",
        "FOR_PCT",
        "BAA10Y",
    ],
]

df = pd.concat([Y, X], axis=1).dropna().iloc[::return_period, :]
Y = df.loc[:, Y.columns]
X = df.loc[:, X.columns]

df.head(10)

### **2.3 Phân tích khám phá dữ liệu (EDA)**
EDA cho phép ta hiểu cấu trúc ẩn của tập dữ liệu bằng cách trực quan hóa các xu hướng, quan hệ và mẫu hình mà mắt người có thể bỏ sót. Với EDA, ta đã có thể bắt đầu rút ra kết luận từ dữ liệu. Khám phá dữ liệu là bước quan trọng trong phân tích dữ liệu. Trong mục này, ta sẽ trình bày phân tích khám phá dữ liệu bằng đồ thị, gồm việc lấy dữ liệu từ dataframe và biểu diễn bằng đồ thị. Bạn cần xem kỹ các đồ thị để hiểu ý nghĩa của chúng.

In [ ]:
df.shape

Dữ liệu này có 243 quan sát và 15 biến.

In [ ]:
df.describe()

Bây giờ ta vẽ các biến dự báo để quan sát hành vi của chúng.

In [ ]:
# print('\033[1m' + 'Fig 5: Yield Curve Prediction' + '\033[0m' )
Y.plot(style=["-", "--", "*"])
pyplot.suptitle(
    "Fig. 5: Yield Curve Prediction",
    fontweight="bold",
    horizontalalignment="right",
    fontsize=15,
)
pyplot.show()

Lưu ý rằng đầu năm 2017, độ lệch của lãi suất kỳ hạn 1 tháng, 5 năm và 30 năm là cao nhưng giảm đều sau đó. Đến năm 2022, lãi suất trung hạn và dài hạn, tức 5 năm và 30 năm, xấp xỉ bằng nhau nhưng lệch rất xa so với lãi suất ngắn hạn. Do đó, các xu hướng chuỗi thời gian này cho thấy hành vi không dừng.

In [ ]:
# Scatterplot Matrix
pyplot.figure(figsize=(15, 15))
scatter_matrix(df, figsize=(15, 16))
pyplot.suptitle(
    "Fig. 6: Scatter Plot Matrix",
    fontweight="bold",
    horizontalalignment="right",
    fontsize=15,
)
pyplot.show()

Từ các biểu đồ phân tán trên, rõ ràng có mối quan hệ tuyến tính mạnh giữa các biến dự báo và các kỳ hạn khác trong tập dữ liệu.

In [ ]:
# correlation
correlation = df.corr()
pyplot.figure(figsize=(16, 16))
pyplot.title("Correlation Matrix")
pyplot.suptitle(
    "Fig. 7: Correlation Plot",
    fontweight="bold",
    horizontalalignment="right",
    fontsize=15,
)
sns.heatmap(correlation, vmax=1, square=True, annot=True, cmap="cubehelix")
pyplot.show()

Biểu đồ tương quan ở trên có thể diễn giải theo cách tương tự biểu đồ phân tán đã bàn trước đó.

Bây giờ ta làm EDA cho các biến dự báo dưới dạng biểu đồ chuỗi thời gian.

Đầu tiên là phân tích chuỗi thời gian kỳ hạn 1 tháng.

In [ ]:
temp_Y = df["1m_pred"]
res = sm.tsa.seasonal_decompose(temp_Y, period=52)
fig = res.plot()
fig.set_figheight(8)
fig.set_figwidth(15)
pyplot.suptitle(
    "Fig. 8: Time Series Trend Analysis for 1 month period",
    fontweight="bold",
    horizontalalignment="right",
    fontsize=15,
)
pyplot.show()

Sau đó là phân tích kỳ hạn 5 năm:

In [ ]:
temp_Y = df["5y_pred"]
res = sm.tsa.seasonal_decompose(temp_Y, period=52)
fig = res.plot()
fig.set_figheight(8)
fig.set_figwidth(15)
pyplot.suptitle(
    "Fig. 9: Time Series Trend Analysis of the 5-Year predicted variable",
    fontweight="bold",
    horizontalalignment="right",
    fontsize=15,
)
pyplot.show()

Cuối cùng là biến kỳ hạn 30 năm

In [ ]:
temp_Y = df["30y_pred"]
res = sm.tsa.seasonal_decompose(temp_Y, period=52)
fig = res.plot()
fig.set_figheight(8)
fig.set_figwidth(15)
pyplot.suptitle(
    "Fig. 10: Time Series Trend Analysis of the 30-Year predicted variable",
    fontweight="bold",
    horizontalalignment="right",
    fontsize=15,
)
pyplot.show()

## **3. Chuẩn bị dữ liệu**

Tập dữ liệu trong bài tập này không cần nhiều bước tiền xử lý, và do đó ta sẽ dùng nguyên trạng. Lưu ý chuẩn bị dữ liệu gồm việc biến đổi dữ liệu, ví dụ co giãn thang đo để loại bỏ độ chệch khi chạy các thuật toán học máy trên tập dữ liệu.

## **4. Mô hình hóa**

###  **4.1 Chia tập dữ liệu thành mẫu huấn luyện và kiểm tra**

Ta sẽ chia dữ liệu thành $80\%$ để huấn luyện (mô hình hóa) và $20\%$ để kiểm định mô hình. Để đánh giá các thuật toán, ta chọn sai số bình phương trung bình vì đây là bài tập hồi quy.

In [ ]:
# split out validation dataset for model validation
test_size = 0.2

seed = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=test_size, random_state=seed
)

Ta dùng kiểm định chéo để kiểm định mô hình khi huấn luyện trên mẫu dữ liệu hạn chế.

In [ ]:
# test options for regression
num_folds = 10
scoring = "neg_mean_squared_error"

### **4.2 Dự báo đường cong lợi suất: khớp đa thức bậc cao**
Hồi quy tuyến tính giả định tồn tại quan hệ tuyến tính giữa biến phụ thuộc và biến độc lập. Để tính đến tính phi tuyến trong tập dữ liệu, ta chọn dùng hồi quy đa thức bằng cách thêm các số hạng đa thức:
$$y = w_1 x + w_2 x^2 + \cdots + w_m x^m + b$$
trong đó $m$ là bậc của đa thức.

Trong nghiên cứu tình huống này, ta sẽ so sánh hiệu suất của hồi quy đa thức bậc 1, bậc 2 và bậc 3. Ta dùng bộ biến đổi `PoynomialFeatures` của scikit-learn để thêm các số hạng bậc hai.

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

pred = lr.predict(X_test)

MSE = mean_squared_error(y_test, pred)

print("linear regression mean squared error:", MSE)

In [ ]:
# Quadratic
poly = PolynomialFeatures(degree=2)
X_train2 = poly.fit_transform(X_train)
X_test2 = poly.fit_transform(X_test)

model = lr.fit(X_train2, y_train)
print(
    "quadratic regression mean squared error:",
    mean_squared_error(y_test, model.predict(X_test2)),
)

# Cubic
poly = PolynomialFeatures(degree=3)
X_train3 = poly.fit_transform(X_train)
X_test3 = poly.fit_transform(X_test)

model = lr.fit(X_train3, y_train)
print(
    "degree 3 polynomial regression mean squared error:",
    mean_squared_error(y_test, model.predict(X_test3)),
)

Rõ ràng hồi quy tuyến tính vượt trội các đa thức bậc cao hơn trong nhiệm vụ dự báo, và do đó ta sẽ dùng kiểm định chéo với nó để so sánh kết quả với một thuật toán học sâu.

### **4.3 Dự báo đường cong lợi suất: kiểm định chéo**
Trong kiểm định chéo $k$ lần, dữ liệu được chia ngẫu nhiên thành $k$ phần và không hoàn lại. Ta huấn luyện dữ liệu trên $k-1$ phần và kiểm định mô hình trên phần còn lại, còn gọi là phần kiểm tra. Phương pháp được lặp lại $k$ lần, nghĩa là ta sẽ có $k$ kết quả. Kiểm định chéo $k$ lần thường dùng để tinh chỉnh mô hình.

Ta ước lượng hiệu suất trung bình $E$ bằng $$E = \frac{1}{k} \sum_{i=1}^k E_i$$

Kiểm định chéo có thể áp dụng cho mọi mô hình, và do đó trước hết ta bàn về mạng nơ-ron ở mục tiếp theo và áp dụng kiểm định chéo vào thuật toán mạng nơ-ron để tinh chỉnh hiệu suất của nó.

### **4.4 Dự báo đường cong lợi suất - Lan truyền ngược**

Ở bài trước, ta đã học chi tiết về lan truyền ngược và ta sẽ thực hành kỹ năng này trong phần này. Sau đó ta sẽ áp dụng một thuật toán mạng nơ-ron nhân tạo (MLP), thuộc thư viện học máy trong `sklearn`.

Trước khi tiếp tục, ta làm quen với một số thuật ngữ thường gặp trong học sâu:
- **Lượt xuôi (Forward pass)**: đồng nghĩa với lan truyền xuôi, tức là đi từ lớp đầu vào đến lớp đầu ra.
- **Lượt ngược (Backward pass)**: đồng nghĩa với lan truyền ngược, tức là đi từ lớp đầu ra về lớp đầu vào.
- **Epoch**: số lần mô hình huấn luyện trên toàn bộ dữ liệu, tức là một epoch tương đương một lượt xuôi và một lượt ngược cho dữ liệu huấn luyện.
- **Vòng lặp (Iterations)**: số lần ta thực hiện lượt xuôi và lượt ngược.
- **Kích thước lô (Batch size)**: tập con của dữ liệu huấn luyện mà ta dùng trong một lượt xuôi và một lượt ngược.

Trong mô hình của ta, ta sẽ dùng mạng nơ-ron hai lớp, trong đó lớp đầu vào có mười hai biến, tức là các biến dự báo. Lớp ẩn có 20 nút để minh họa, và lớp đầu ra có ba nút tương ứng với các biến được dự báo. Vì vậy ta bắt đầu bằng việc định nghĩa số nút ở mỗi lớp.

In [ ]:
from nnv import NNV

print("\033[1m" + "Fig 12: A Two Layer Neural Network" + "\033[0m")
layersList = [
    {"title": "inputs \n + bias", "units": 12, "color": "darkBlue"},
    {"title": "hidden layer", "units": 20},
    # {"title":"$f(z)$", "units": 1, "edges_color":"red", "edges_width":2},
    {"title": "output", "units": 3, "color": "darkBlue"},
]

NNV(layersList).render(save_to_file="my_example.png")
pyplot.show()

In [ ]:
num_input = 12
num_hidden = 20
num_output = 3

Tiếp theo, ta khởi tạo ngẫu nhiên giá trị của các trọng số và số hạng chệch. Ta bắt đầu bằng việc khởi tạo trọng số từ đầu vào đến lớp ẩn.

In [ ]:
W_x_h = np.random.randn(num_input, num_hidden)
b_h = np.zeros((1, num_hidden))

Sau đó, ta khởi tạo trọng số và số hạng chệch từ lớp ẩn đến lớp đầu ra.

In [ ]:
W_h_y = np.random.randn(num_hidden, num_output)
b_y = np.zeros((1, num_output))

Bây giờ ta có thể định nghĩa hàm kích hoạt và đạo hàm của nó, trong trường hợp này là hàm sigmoid.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def sigmoid_derivative(z):
    return np.exp(-z) / ((1 + np.exp(-z)) ** 2)

Như ở phần trước, ta viết các hàm lan truyền xuôi và lan truyền ngược.

In [ ]:
def forward_prop(X, W_x_h, W_h_y):
    z_1 = np.dot(X, W_x_h) + b_h
    a_1 = sigmoid(z_1)
    z_2 = np.dot(a_1, W_h_y) + b_y
    y_hat = sigmoid(z_2)

    return z_1, a_1, z_2, y_hat


def back_prop(y_hat, z_1, a_1, z_2):
    delta2 = np.multiply(-(y_train - y_hat), sigmoid_derivative(z_2))
    dJ_d_W_h_y = np.dot(a_1.T, delta2)
    delta1 = np.dot(delta2, W_h_y.T) * sigmoid_derivative(z_1)
    dJ_d_W_x_h = np.dot(X_train.T, delta1)

    return dJ_d_W_x_h, dJ_d_W_h_y

Hàm chi phí ta dùng là sai số bình phương như bên dưới:

In [ ]:
def cost_function(y, y_pred):
    return 0.5 * np.sum((y - y_pred) ** 2)

Trước khi huấn luyện mô hình, ta đặt tốc độ học và số vòng lặp.

In [ ]:
alpha = 0.01
num_iterations = 10000

Bây giờ ta huấn luyện mạng nơ-ron.

In [ ]:
cost = []

for i in range(num_iterations):
    z_1, a_1, z_2, y_hat = forward_prop(X_train, W_x_h, W_h_y)
    dJ_d_W_x_h, dJ_d_W_h_y = back_prop(y_hat, z_1, a_1, z_2)

    # update weights
    W_x_h = W_x_h - alpha * dJ_d_W_x_h
    W_h_y = W_h_y - alpha * dJ_d_W_h_y

    # compute cost
    c = cost_function(y_train, y_hat)

    cost.append(c)

Đồ thị của hàm chi phí được trình bày bên dưới:

In [ ]:
pyplot.grid()
pyplot.plot(range(num_iterations), cost)

pyplot.title("Cost Function")
pyplot.xlabel("Training Iterations")
pyplot.ylabel("Cost")
pyplot.suptitle(
    "Fig. 13: Cost Function Plot",
    fontweight="bold",
    horizontalalignment="right",
    fontsize=15,
)
pyplot.show()

So với các mô hình hồi quy ở mục đầu của phần này, mô hình mạng nơ-ron hoạt động kém. Ta luôn có thể cải thiện hiệu suất này bằng cách tăng số nút ở lớp ẩn hoặc tăng số lớp ẩn để cho phép mô hình hóa dữ liệu phức tạp.

Bây giờ ta đã hiểu rõ cách mạng nơ-ron hoạt động, hãy cài đặt một thuật toán mạng nơ-ron gọi là **Perceptron nhiều lớp** (Multilayer Perceptron) để dự báo đường cong lợi suất.

Ta sẽ dùng Perceptron nhiều lớp (MLP) làm ví dụ về mạng nơ-ron sâu kết nối đầy đủ.

Như đã bàn ở trên, thuật toán MLP theo các bước sau:

1. Với một lớp đầu vào, dữ liệu huấn luyện được lan truyền xuôi qua mạng nơ-ron sâu để có đầu ra.
2. Sau đó, tính mất mát để kiểm tra nó đã ở mức cực tiểu chưa.
3. Mất mát được lan truyền ngược và mô hình được cập nhật.

Cuối cùng, khi ta hài lòng với đầu ra của ba bước trên, ta dùng lan truyền xuôi để có đầu ra cuối cùng.

Bên dưới là minh họa việc dùng kiểm định chéo và lan truyền ngược trong dự báo đường cong lợi suất.

In [ ]:
# test options for regression
num_folds = 10
scoring = "neg_mean_squared_error"

In [ ]:
# spot check the algorithms
models = []

# Linear Regression Algorithm
models.append(("LR", LinearRegression()))

# Deep Learning Algorithm
models.append(("MLP", MLPRegressor()))

In [ ]:
kfold_results = []
names = []
validation_results = []
train_results = []
for name, model in models:
    kfold = KFold(n_splits=num_folds)
    # converted mean square error to positive. The lower the better
    cv_results = -1 * cross_val_score(
        model, X_train, y_train, cv=kfold, scoring=scoring
    )
    kfold_results.append(cv_results)
    names.append(name)

    # Finally we Train on the full period and test against validation
    res = model.fit(X_train, y_train)
    validation_result = np.mean(np.square(res.predict(X_test) - y_test))
    validation_results.append(validation_result)
    train_result = np.mean(np.square(res.predict(X_train) - y_train))
    train_results.append(train_result)

    msg = (
        "%s: \nAverage CV error: %s \nStd CV Error: (%s) \nTraining Error:\n%s \nTest Error:\n%s"
        % (
            name,
            str(cv_results.mean()),
            str(cv_results.std()),
            str(train_result),
            str(validation_result),
        )
    )
    print(msg)
    print("----------")

Bây giờ ta so sánh hiệu suất của hai thuật toán.

In [ ]:
# compare algorithms
names = ["Linear Regression", "Multilinear Perceptron"]
fig = pyplot.figure()
# print('\033[1m' + 'Fig 14: Box Plot Comparing Model Performance' + '\033[0m' )
fig.suptitle("Algorithm Comparison")
ax = fig.add_subplot(111)
pyplot.boxplot(kfold_results)
ax.set_xticklabels(names)
fig.set_size_inches(15, 8)
pyplot.ylabel("Model Errors")
pyplot.xlabel("Models")
pyplot.suptitle(
    "Fig. 14: Box Plot Comparing Model Performance",
    fontweight="bold",
    horizontalalignment="right",
    fontsize=15,
)
pyplot.show()

In [ ]:
# compare algorithms
fig = pyplot.figure()

ind = np.arange(len(names))  # the x locations for the groups
width = 0.35  # the width of the bars

fig.suptitle("Algorithm Comparison")
ax = fig.add_subplot(111)
pyplot.bar(
    ind - width / 2, [x.mean() for x in train_results], width=width, label="Train Error"
)
pyplot.bar(
    ind + width / 2,
    [x.mean() for x in validation_results],
    width=width,
    label="Validation Error",
)
fig.set_size_inches(15, 8)
pyplot.legend()
pyplot.ylabel("Error")
pyplot.xlabel("Models")
ax.set_xticks(ind)
ax.set_xticklabels(names)
pyplot.suptitle(
    "Fig. 15: Bar Plot Comparing Training and Validation Error",
    fontweight="bold",
    horizontalalignment="right",
    fontsize=15,
)
pyplot.show()

Mô hình hồi quy tuyến tính có ít sai số hơn MLP dù đơn giản. Để cải thiện hiệu suất của bộ hồi quy MLP, ta tìm các tham số tốt nhất bằng tìm kiếm lưới như bên dưới.

In [ ]:
# Grid search : `MLPRegressor`
"""
hidden_layer_sizes : tuple, length = n_layers - 2, default (100,)
    The ith element represents the number of neurons in the ith
    hidden layer.
"""
param_grid = {"hidden_layer_sizes": [(20,), (50,), (20, 20), (20, 30, 20)]}
model = MLPRegressor()
kfold = KFold(n_splits=num_folds)
grid = GridSearchCV(estimator=model, param_grid=param_grid, scoring=scoring, cv=kfold)
grid_result = grid.fit(X_train, y_train)
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))
means = grid_result.cv_results_["mean_test_score"]

In [ ]:
# prepare model
model = MLPRegressor(hidden_layer_sizes=(20, 30, 20))
model.fit(X_train, y_train)

In [ ]:
# estimate accuracy on validation set
from sklearn.metrics import mean_squared_error, r2_score

predictions = model.predict(X_test)
mse_MLP = mean_squared_error(y_test, predictions)
r2_MLP = r2_score(y_test, predictions)

# prepare model
model_2 = LinearRegression()
model_2.fit(X_train, y_train)
predictions_2 = model_2.predict(X_test)

mse_OLS = mean_squared_error(y_test, predictions_2)
r2_OLS = r2_score(y_test, predictions_2)
print("MSE Regression = %f, MSE MLP = %f" % (mse_OLS, mse_MLP))
print("R2 Regression = %f, R2 MLP = %f" % (r2_OLS, r2_MLP))

In [ ]:
train_size = int(len(X) * (1 - test_size))
X_train, X_test = X[0:train_size], X[train_size : len(X)]
y_train, y_test = Y[0:train_size], Y[train_size : len(X)]

modelMLP = MLPRegressor(hidden_layer_sizes=(50,))
modelOLS = LinearRegression()
model_MLP = modelMLP.fit(X_train, y_train)
model_OLS = modelOLS.fit(X_train, y_train)

Y_predMLP = pd.DataFrame(
    model_MLP.predict(X_test), index=y_test.index, columns=y_test.columns
)

Y_predOLS = pd.DataFrame(
    model_OLS.predict(X_test), index=y_test.index, columns=y_test.columns
)

Ta so sánh hai dự báo bên dưới:

In [ ]:
pd.DataFrame(
    {
        "Actual : 1m": y_test.loc[:, "1m_pred"],
        "Prediction MLP 1m": Y_predMLP.loc[:, "1m_pred"],
        "Prediction OLS 1m": Y_predOLS.loc[:, "1m_pred"],
    }
).plot(figsize=(10, 5))
pyplot.title("1 month rate prediction")
pyplot.ylabel("Yield Rates")
pyplot.suptitle(
    "Fig. 16: 1 Month Rate Prediction",
    fontweight="bold",
    horizontalalignment="right",
    fontsize=15,
)
pyplot.show

In [ ]:
pd.DataFrame(
    {
        "Actual : 5yr": y_test.loc[:, "5y_pred"],
        "Prediction MLP 5yr": Y_predMLP.loc[:, "5y_pred"],
        "Prediction OLS 5yr": Y_predOLS.loc[:, "5y_pred"],
    }
).plot(figsize=(10, 5))
pyplot.title("5 years rate prediction")
pyplot.ylabel("Yield Rates")
pyplot.suptitle(
    "Fig. 17: 5 Years Prediction",
    fontweight="bold",
    horizontalalignment="right",
    fontsize=15,
)
pyplot.show

In [ ]:
pd.DataFrame(
    {
        "Actual : 30yr": y_test.loc[:, "30y_pred"],
        "Prediction MLP 30yr": Y_predMLP.loc[:, "30y_pred"],
        "Prediction OLS 30yr": Y_predOLS.loc[:, "30y_pred"],
    }
).plot(figsize=(10, 5))
pyplot.title("30 years rate prediction")
pyplot.ylabel("Yield Rates")
pyplot.suptitle(
    "Fig. 18: 30 Year Prediction",
    fontweight="bold",
    horizontalalignment="right",
    fontsize=15,
)
pyplot.show

Từ các biểu đồ trên, mô hình hồi quy tuyến tính có vẻ cho dự báo lợi suất tốt hơn so với mô hình ANN.

## **5. Kết luận**

Trong bài này, ta đã áp dụng hồi quy tuyến tính thông thường và một mô hình học sâu để dự báo đường cong lợi suất ở cả ngắn hạn, trung hạn và dài hạn. Dù mô hình hồi quy tuyến tính vượt trội mô hình MLP trong bài tập này, có nhiều bài toán mà mô hình học sâu sẽ cho kết quả tốt hơn, và do đó nên thử một vài mô hình trước khi chọn một mô hình. Ta cũng đã thấy luôn có thể cải thiện hiệu suất của mạng nơ-ron bằng cách thực hiện tìm kiếm lưới để chọn tham số tối ưu.

Đầu ra của mô hình học sâu có thể được cải thiện bằng cách tìm siêu tham số phù hợp và tăng số lớp của mô hình. Ở bài tiếp theo, ta sẽ xem các thuật toán tối ưu hóa trong học sâu, những thách thức chúng gặp phải, chúng ảnh hưởng thế nào đến đầu ra của mô hình, và cách giải quyết các thách thức này.

**Tài liệu tham khảo**

1. Bernadell, Carlos, et al. "Yield Curve Prediction for the Strategic Investor." European Central Bank, *Working Paper Series*, no. 472, 2005.
2. Bluwstein, Kristina, et al. "Credit Growth, the Yield Curve and Financial Crisis Prediction: Evidence from a Machine Learning Approach." (2021).
3. Hyndman, Cody. "Arbitrage-Free Yield Curve and Bond Price Forecasting by Deep Neural Networks." *YouTube*, uploaded by Fields Institute, 29 Sept. 2021.
4. Arnold, Ludovic, et al. "An Introduction to Deep Learning." European Symposium on Artificial Neural Networks (ESANN), 2011.